# German-to-English Translation with Flan-T5

This notebook applies Google's Flan-T5 — a sequence-to-sequence encoder-decoder transformer instruction-tuned on a diverse mixture of tasks — to translate German literary text into English. Translation quality is evaluated using BLEU scores. A scaling experiment compares performance across three model sizes: `flan-t5-small`, `flan-t5-base`, and `flan-t5-large`.

Unlike decoder-only causal models, Flan-T5 uses a full encoder-decoder architecture: the encoder processes the input bidirectionally and the decoder autoregressively generates the output conditioned on the encoded representation.

The `transformers`, `datasets`, and `evaluate` packages from Hugging Face are required. Install with `pip install transformers datasets evaluate sentencepiece` if not already present.

In [1]:
# pip install transformers

In [2]:
# pip install datasets

## 1. Tokenizer

The shared Flan-T5 tokenizer (a SentencePiece unigram model) is used across all model sizes. Unlike BERT's WordPiece vocabulary, it handles multilingual input natively, which is essential for a translation task involving German text.

In [3]:
# Load the Flan-T5 tokenizer (shared across all model sizes)

from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

print("Vocabulary size: ", tokenizer.vocab_size)
tokenized = tokenizer(["The little black cat sleeps in the window", 
                       "Le petit chat noir dort dans la fenêtre"], padding='longest')
print(tokenized)

/home/users/gcc14/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Vocabulary size:  32000
{'input_ids': [[37, 385, 1001, 1712, 2085, 7, 16, 8, 2034, 1], [312, 4561, 3582, 9691, 5048, 247, 50, 25301, 1, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]]}


## 2. Model — `flan-t5-small`

The smallest Flan-T5 variant is loaded in 16-bit float precision (`torch_dtype=torch.float16`) to reduce memory footprint. All three model sizes share the same architecture; they differ only in the number of layers, heads, and embedding dimension.

**Memory calculation:** Each parameter occupies 2 bytes (16-bit). Total memory = `num_parameters × 2 bytes`.

In [4]:
# Load flan-t5-small in 16-bit precision
import torch
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small", torch_dtype=torch.float16)
print(model)

2024-11-14 10:58:52.603028: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

Examining the model's memory footprint from its parameter count:

In [5]:
tot_param = sum(p.numel() for p in model.parameters())

bytes_total = 2 * tot_param
kilobytes = bytes_total / 1024
megabytes = kilobytes / 1024

print(f'Parameters:   {tot_param:,}')
print(f'Memory (KB):  {kilobytes:,.0f} KB')
print(f'Memory (MB):  {megabytes:.1f} MB')

150314.75

At 16-bit (2-byte) precision, storing the model requires approximately **150,315 KB (~147 MB)**. Each parameter occupies exactly 2 bytes, so total storage = `num_parameters × 2 bytes`.

## 3. Prompting for Translation

Flan-T5 was pretrained on multiple tasks. Without a task-specific prompt, the model does not produce translations — it defaults to other behaviors from its pretraining mixture. The demonstration below shows the effect of prompting.

In [6]:
# Demonstrate model.generate without prompting (does not translate)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

input_text = ["The little black cat sleeps in the window", "The dog runs in the field"]
encoded = tokenizer(input_text, return_tensors="pt", padding="longest").to(device)

outputs = model.generate(**encoded, max_new_tokens=100)
print(outputs)

for out in outputs:
    print(tokenizer.decode(out, skip_special_tokens=True))

tensor([[   0,   37, 1712,   19,    3,    9, 1712,    5,    1,    0],
        [   0,   37, 1782,   19, 1180,   16,    8, 1057,    5,    1]],
       device='cuda:0')
The cat is a cat.
The dog is running in the field.


Prefixing inputs with `"Translate English into French: "` activates the translation behavior learned during Flan-T5's instruction tuning. The same principle applies for German-to-English translation.

In [7]:
# Demonstrate prompted translation (English → French)

input_text = ["The little black cat sleeps in the window", "The dog runs in the field"]
prompt = "Translate English into French: "
prompted_text = [prompt + in_text for in_text in input_text]
encoded = tokenizer(prompted_text, return_tensors="pt", padding="longest").to(device)

outputs = model.generate(**encoded, max_new_tokens=100)

for out in outputs:
    print(tokenizer.decode(out, skip_special_tokens=True))

La petite cat noir s'est en vertu de la fenêtre.
Le chien s'est en l'égard de la field.


## 4. Dataset

500 sentence pairs are loaded from the [OPUS Books](https://huggingface.co/datasets/opus_books) `de-en` parallel corpus — a literary translation dataset including a German-English edition of *Jane Eyre*. Each example contains the original German text and a reference English translation.

In [8]:
# Load 500 German-English sentence pairs from OPUS Books (Jane Eyre)

from datasets import load_dataset
from torch.utils.data import Dataset

class OpusDataset(Dataset):
    def __init__(self, dataset_stream, num_examples):
        # Convert streaming dataset to list for random access
        self.examples = list(dataset_stream.take(num_examples))
        
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

# Load the dataset in streaming mode
dataset_stream = load_dataset(
    "opus_books",
    "de-en",
    split="train",
    streaming=True
)

# Create instance of custom dataset
dataset = OpusDataset(dataset_stream, num_examples=500)

# Print a few examples to verify
for i in range(100, 103):
    print(f"\nExample {i+1}:")
    print(f"German: {dataset[i]['translation']['de']}")
    print(f"English: {dataset[i]['translation']['en']}")
print(f"\nTotal examples loaded: {len(dataset)}")


Example 101:
German: Dieser Vorwurf meiner Abhängigkeit war in meinen Ohren fast zum leeren, bedeutungslosen Singsang geworden, sehr schmerzlich und bedrückend, aber nur halb verständlich.
English: This reproach of my dependence had become a vague sing-song in my ear: very painful and crushing, but only half intelligible.

Example 102:
German: Nun fiel auch Miß Abbot ein: »Und Sie sollten auch nicht denken, daß Sie mit den Fräulein Reed und Mr. Reed auf gleicher Stufe stehen, weil Mrs. Reed Ihnen gütig erlaubt, mit ihren Kindern erzogen zu werden.
English: Miss Abbot joined in-- "And you ought not to think yourself on an equality with the Misses Reed and Master Reed, because Missis kindly allows you to be brought up with them.

Example 103:
German: Diese werden einmal ein großes Vermögen haben, und Sie sind arm. Sie müssen demütig und bescheiden sein und versuchen, sich den andern angenehm zu machen.«
English: They will have a great deal of money, and you will have none: it is your pl

## 5. German → English Translation

All 500 German sentences are translated using the `flan-t5-small` model. Each sentence is independently prompted and decoded. Three examples are printed comparing the model output to the reference English translation.

In [9]:
# Translate all 500 German sentences with flan-t5-small
translated = []

for i in range(len(dataset)):
    prompt = "Translate German into English: "
    german = dataset[i]['translation']['de']
    prompted_text = [prompt + german]
    encoded = tokenizer(prompted_text, return_tensors="pt", padding="longest").to(device)
    outputs = model.generate(**encoded, max_new_tokens=100)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translated.append(translation)

    if((i+1) % 10 == 0):
        print(f"Translated {i+1} phrases from German to English")

print("")
print("Example 1")
print(f"German text: {dataset[123]['translation']['de']}")
print(f"English text: {dataset[123]['translation']['en']}")
print(f"Translated text: {translated[123]}")
print("")
print("Example 2")
print(f"German text: {dataset[400]['translation']['de']}")
print(f"English text: {dataset[400]['translation']['en']}")
print(f"Translated text: {translated[400]}")
print("")
print("Example 3")
print(f"German text: {dataset[300]['translation']['de']}")
print(f"English text: {dataset[300]['translation']['en']}")
print(f"Translated text: {translated[300]}")

Translated 10 phrases from German to English
Translated 20 phrases from German to English
Translated 30 phrases from German to English
Translated 40 phrases from German to English
Translated 50 phrases from German to English
Translated 60 phrases from German to English
Translated 70 phrases from German to English
Translated 80 phrases from German to English
Translated 90 phrases from German to English
Translated 100 phrases from German to English
Translated 110 phrases from German to English
Translated 120 phrases from German to English
Translated 130 phrases from German to English
Translated 140 phrases from German to English
Translated 150 phrases from German to English
Translated 160 phrases from German to English
Translated 170 phrases from German to English
Translated 180 phrases from German to English
Translated 190 phrases from German to English
Translated 200 phrases from German to English
Translated 210 phrases from German to English
Translated 220 phrases from German to Engli

## 6. BLEU Score Evaluation

[BLEU](https://en.wikipedia.org/wiki/BLEU) (Bilingual Evaluation Understudy) measures n-gram overlap between predicted and reference translations, on a 0–1 scale (this implementation) or 0–100. It is an imperfect but widely used automatic metric. A score of 1.0 is not expected — many valid translations differ in wording from the reference.

In [10]:
# Demonstrate BLEU score calculation

import evaluate

predictions = ["the black cat is sleeping in the sun by the window", 
               "the dog runs in the field while it rains"]
references = [["the black cat is sleeps on the sun by the window"], 
              ["the dog run in the field while it rain"]]

bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

{'bleu': 0.5555238068023582, 'precisions': [0.8, 0.6666666666666666, 0.5, 0.35714285714285715], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 20, 'reference_length': 20}


BLEU scores for the `flan-t5-small` translations against the reference English text:

In [11]:
# Compute BLEU score for flan-t5-small translations
english = []

for i in range(len(dataset)):
    english.append([dataset[i]['translation']['en']])

results = bleu.compute(predictions=translated, references=english)
print(results)

{'bleu': 0.06793617635586857, 'precisions': [0.33071657351001693, 0.09985087207417494, 0.03799504121155264, 0.016977340447647427], 'brevity_penalty': 1.0, 'length_ratio': 1.0599081408506956, 'translation_length': 15923, 'reference_length': 15023}


## 7. Scaling Study: flan-t5-base and flan-t5-large


The same 500-sentence translation task is repeated with `flan-t5-base` and `flan-t5-large`. All other settings (prompt, `max_new_tokens`, dataset) are identical, isolating model scale as the only variable.

In [12]:
# Translate with flan-t5-base and flan-t5-large; compute BLEU scores

# Base model
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base", torch_dtype=torch.float16)
model = model.to(device)
translated_base = []

print("Base model:")

for i in range(len(dataset)):
    prompt = "Translate German into English: "
    german = dataset[i]['translation']['de']
    prompted_text = [prompt + german]
    encoded = tokenizer(prompted_text, return_tensors="pt", padding="longest").to(device)
    outputs = model.generate(**encoded, max_new_tokens=100)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translated_base.append(translation)

    if((i+1) % 50 == 0):
        print(f"Translated {i+1} phrases from German to English")
        
results_base = bleu.compute(predictions=translated_base, references=english)
print(results_base)
print("")
# Large model

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large", torch_dtype=torch.float16)
model = model.to(device)
translated_large = []

print("Large model:")

for i in range(len(dataset)):
    prompt = "Translate German into English: "
    german = dataset[i]['translation']['de']
    prompted_text = [prompt + german]
    encoded = tokenizer(prompted_text, return_tensors="pt", padding="longest").to(device)
    outputs = model.generate(**encoded, max_new_tokens=100)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    translated_large.append(translation)

    if((i+1) % 50 == 0):
        print(f"Translated {i+1} phrases from German to English")
        
results_large = bleu.compute(predictions=translated_large, references=english)
print(results_large)

Base model:
Translated 50 phrases from German to English
Translated 100 phrases from German to English
Translated 150 phrases from German to English
Translated 200 phrases from German to English
Translated 250 phrases from German to English
Translated 300 phrases from German to English
Translated 350 phrases from German to English
Translated 400 phrases from German to English
Translated 450 phrases from German to English
Translated 500 phrases from German to English
{'bleu': 0.12426410934088251, 'precisions': [0.45540809555408096, 0.17419045005488473, 0.07754580315296122, 0.038761400411885846], 'brevity_penalty': 1.0, 'length_ratio': 1.003128536244425, 'translation_length': 15070, 'reference_length': 15023}

Large model:
Translated 50 phrases from German to English
Translated 100 phrases from German to English
Translated 150 phrases from German to English
Translated 200 phrases from German to English
Translated 250 phrases from German to English
Translated 300 phrases from German to En